In [86]:
import os 
print(os.getcwd())

C:\Users\Shawn\PROJECT\Stocksense\Reddit\reddit cleaning


In [87]:
import glob

all_csv = glob.glob("*.csv")
print(all_csv)

['AAPL.csv', 'AMZN.csv', 'ARK.csv', 'ATER.csv', 'GOOGL.csv', 'MSFT.csv', 'RUI.csv', 'TSLA.csv', 'VXRT.csv']


In [88]:
import re
import nltk
import numpy as np
import pandas as pd
from nltk import word_tokenize
from nltk.tokenize import RegexpTokenizer
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords

# text cleaning
def text_lower(text):
    return text.lower()

def remove_hashtag_mentions_urls(text):
    return re.sub(r"(?:\@|\#|https?\://)\S+", "", text)

def remove_emoji(text):
    emoji_pattern = re.compile("["
    u"\U0001F600-\U0001F64F"  # emoticons
    u"\U0001F300-\U0001F5FF"  # symbols & pictographs
    u"\U0001F680-\U0001F6FF"  # transport & map symbols
    u"\U0001F1E0-\U0001F1FF"  # flags (iOS)
    u"\U00002702-\U000027B0"
    u"\U000024C2-\U0001F251"
    "]+", flags=re.UNICODE)

    return emoji_pattern.sub(r'', text)

def tokenization(text):
    word_tokenizer = RegexpTokenizer(r'[-\'\w]+')
    tokenized_text = word_tokenizer.tokenize(text)
    return tokenized_text

In [89]:
def clean_reddit(filepath):
    csvdf = pd.read_csv(filepath)
    
    #cleaning reddit
    reddit_text = pd.DataFrame()
    reddit_text[['text','date','likes','subreddit']] = csvdf
    
    for index,row in reddit_text.iterrows():

        individual_text = row[0]
        individual_text = word_tokenize(individual_text)
        individual_text_lower = [w.lower() for w in individual_text]
        individual_text_only = [w for w in individual_text_lower if re.search('^[a-z]+$',w)]
        individual_text_no_emoji =  [w for w in individual_text_only if remove_emoji(w)]


        individual_text_sentence = ' '.join(individual_text_no_emoji)

        reddit_text.at[index,'cleaned_text'] = individual_text_sentence
            
    return reddit_text

In [90]:
# test = clean_reddit(all_csv[0])
# test.head()

In [91]:
for file in all_csv:
    clean_csv = clean_reddit(file)
    filename = "clean/cleaned_" + file
    print(filename,"done")
    clean_csv.to_csv(filename)

clean/cleaned_AAPL.csv done
clean/cleaned_AMZN.csv done
clean/cleaned_ARK.csv done
clean/cleaned_ATER.csv done
clean/cleaned_GOOGL.csv done
clean/cleaned_MSFT.csv done
clean/cleaned_RUI.csv done
clean/cleaned_TSLA.csv done
clean/cleaned_VXRT.csv done


In [2]:
import pandas as pd
test = pd.read_csv("clean/cleaned_AAPL.csv")
test = test.dropna()
test.info()
test = test["cleaned_text"].tolist()
print(test[:5])

<class 'pandas.core.frame.DataFrame'>
Int64Index: 28988 entries, 0 to 29001
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   Unnamed: 0    28988 non-null  int64 
 1   text          28988 non-null  object
 2   date          28988 non-null  object
 3   likes         28988 non-null  int64 
 4   subreddit     28988 non-null  object
 5   cleaned_text  28988 non-null  object
dtypes: int64(2), object(4)
memory usage: 1.5+ MB
['which stocks or positions are the clear winners of i know that last year apple tesla and aark did amazing', 'if apple tesla and aark where massive winning stocks in what the the clear winners so far in', 'can someone please rate my portfolio pypl aapl jnj wmt pltr uber', 'palantir pltr and job postings preferred experience facebook apple at amp t', 'events like pltr demo day i a entry level investor and am starting to get familiar with stock events i use tos and tos calendar has been useful as i was

In [8]:
pip install happytransformer

  Using cached tqdm-4.62.3-py2.py3-none-any.whl (76 kB)
  Attempting uninstall: tqdm
    Found existing installation: tqdm 4.59.0
    Uninstalling tqdm-4.59.0:
      Successfully uninstalled tqdm-4.59.0
  Attempting uninstall: fsspec
    Found existing installation: fsspec 0.9.0
    Uninstalling fsspec-0.9.0:
      Successfully uninstalled fsspec-0.9.0
Note: you may need to restart the kernel to use updated packages.


In [9]:

from happytransformer import HappyTextClassification 
happy_tc = HappyTextClassification("BERT", "ProsusAI/finbert", num_labels=3)
result = happy_tc.classify_text("Tesla's stock just increased by 20%")
print(result)
print(result.label)
print(result.score)

10/13/2021 01:34:46 - INFO - happytransformer.happy_transformer -   Using model: cpu


TextClassificationResult(label='positive', score=0.9291105270385742)
positive
0.9291105270385742


In [10]:
result = happy_tc.classify_text("The price of gold just dropped by 5%")
print(result.label)
print(result.score)

negative
0.8852561712265015


In [12]:
import csv

cases= [("Wow I love using BERT for text classification", 0), ("I hate NLP", 1)]

with open("train.csv", 'w', newline='') as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(["text", "label"])
        for case in cases:
            writer.writerow([case[0], case[1]])
            
happy_tc = HappyTextClassification(model_type="DISTILBERT", model_name="distilbert-base-uncased", num_labels=2)

happy_tc.train("train.csv")

Some weights of the model checkpoint at distilbert-base-uncased were not used when initializing DistilBertForSequenceClassification: ['vocab_layer_norm.bias', 'vocab_layer_norm.weight', 'vocab_projector.weight', 'vocab_projector.bias', 'vocab_transform.weight', 'vocab_transform.bias']
- This IS expected if you are initializing DistilBertForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DistilBertForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['pre_classifier.weight', 'classifier.bias', 'classifier

Step,Training Loss




Training completed. Do not forget to share your model on huggingface.co/models =)


